# E-Commerce Sales Analytics - Exploratory Analysis
Dataset: Amazon India sales report (Mar-Jun 2022). Run this notebook from the `Python/` folder.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys; sys.path.append("..")
from Python.clean_data import clean

sns.set_theme(style="whitegrid")
pd.options.display.float_format = "{:,.2f}".format

## 1. Load raw data & first look

In [ ]:
raw = pd.read_csv("../Dataset/Amazon Sale Report.csv", low_memory=False)
print(raw.shape)
raw.head()

In [ ]:
raw.info()
raw.isna().sum().sort_values(ascending=False).head(10)

## 2. Clean data
The cleaning logic lives in `clean_data.py` so the notebook, the dashboard and the report all use the same rules (dedupe, parse dates, fix state spellings, add flags).

In [ ]:
df = clean("../Dataset/Amazon Sale Report.csv")
print(df.shape)
df.head()

## 3. KPIs

In [ ]:
sales = df[df.is_valid_sale]
kpi = {
    "Total revenue (INR)": sales.revenue.sum(),
    "Orders (rows)": len(df),
    "Unique order IDs": df.order_id.nunique(),
    "Units sold": sales.qty.sum(),
    "Avg order value": sales.revenue.mean(),
    "Cancel rate %": df.is_cancelled.mean()*100,
    "Return rate %": df.is_returned.mean()*100,
}
pd.Series(kpi)

## 4. Sales trend

In [ ]:
daily = sales.groupby("date").revenue.sum()
ax = daily.plot(figsize=(12,4), label="daily")
daily.rolling(7).mean().plot(ax=ax, label="7-day avg")
ax.set_title("Daily revenue"); ax.legend(); plt.show()

In [ ]:
sales.groupby("month", observed=True).revenue.sum().plot(kind="bar", figsize=(6,4), title="Monthly revenue")
plt.show()
# NOTE: March holds only one day (31 Mar) - do not compare it with other months.

## 5. Product analysis

In [ ]:
cat = sales.groupby("category", observed=True).agg(revenue=("revenue","sum"), units=("qty","sum"))
cat["share %"] = cat.revenue/cat.revenue.sum()*100
cat.sort_values("revenue", ascending=False)

In [ ]:
top_styles = sales.groupby("style").revenue.sum().nlargest(10)
top_styles.sort_values().plot(kind="barh", figsize=(8,4), title="Top 10 styles")
plt.show()

In [ ]:
size_order = ["XS","S","M","L","XL","XXL","3XL","4XL","5XL","6XL","Free"]
sales.groupby("size", observed=True).qty.sum().reindex(size_order).plot(kind="bar", figsize=(8,4), title="Units by size")
plt.show()

## 6. Geography

In [ ]:
state = sales.groupby("ship_state", observed=True).revenue.sum().nlargest(10)
state.sort_values().plot(kind="barh", figsize=(8,4), title="Top 10 states by revenue")
plt.show()

## 7. Operations: cancellations, returns, fulfilment

In [ ]:
df.status.value_counts().sort_values().plot(kind="barh", figsize=(8,5), title="Orders by status")
plt.show()

In [ ]:
df.groupby("fulfilment", observed=True)[["is_cancelled","is_returned"]].mean().mul(100).round(2)

In [ ]:
df.groupby("ship_service_level", observed=True)[["is_cancelled","is_returned"]].mean().mul(100).round(2)

## 8. Promotions and B2B

In [ ]:
print(df.groupby("has_promotion").agg(orders=("order_id","count"), cancel_rate=("is_cancelled","mean")))
print(sales.groupby("b2b").agg(orders=("order_id","count"), aov=("revenue","mean")))

## 9. Weekday pattern

In [ ]:
order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
sales.groupby("weekday", observed=True).revenue.sum().reindex(order).plot(kind="bar", figsize=(8,4), title="Revenue by weekday")
plt.show()

## 10. Key insights
Write your findings here after running the cells (top category, top state, cancel rate, best fulfilment channel, growth trend). Copy them into `README.md` and `Report.pdf`.